# AI Multi-Agent Equity Research Platform — End-to-End Demo

이 노트북은 `src/` 아래 각 컴포넌트(① Data Collector ~ ⑩ PDF Report, 그리고 Tool-calling Agent)를 순서대로 실제로 돌려보는 데모다. 셀을 위에서부터 순서대로 실행하면 각 단계 결과를 바로 확인할 수 있다.

**사전 준비**: `pip install -r requirements.txt` 후, `DART_API_KEY`/`NAVER_CLIENT_ID`/`NAVER_CLIENT_SECRET`/`FINNHUB_API_KEY`/`GEMINI_API_KEY`를 프로젝트 루트의 `.env` 또는 홈 디렉토리(`~/.env`)에 설정해야 한다. (`.env.example` 참고)

**대상 기업**: 아래 셀에서 `TICKER`/`COMPANY_NAME`만 바꾸면 다른 종목으로 전체 파이프라인을 다시 돌릴 수 있다. 미국 종목을 권장한다 — 실적발표 대본(`transcript.py`)은 한국 종목을 지원하지 않는다(README/독스트링 참고). 단, 대본은 fool.com의 "최근 발표" 목록에서만 검색되므로(`find_transcript_url`), 시간이 지나 해당 분기가 목록에서 밀려나면 다른 최근 실적발표 종목으로 바꿔야 할 수 있다 — 대본 관련 셀들은 못 찾으면 에러 없이 건너뛰도록 만들어뒀다.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

TICKER = "TSM"
COMPANY_NAME = "TSMC"

from src.config import market_of
MARKET = market_of(TICKER)
print(f"{COMPANY_NAME} ({TICKER}) — market: {MARKET}")

TSMC (TSM) — market: US


## ① Data Collector

공시(`dart.py`/`sec.py`), 시세·기업정보(`market.py`), 뉴스(`news.py`)를 수집한다. 시장(KR/US)에 따라 자동으로 적절한 수집기를 쓴다.

In [2]:
from src.collector import dart, sec, market, news

info = market.get_company_info(TICKER)
print("현재가:", info.get("currentPrice"), info.get("currency"))
print("시가총액:", info.get("marketCap"))

if MARKET == "KR":
    stock_code = TICKER.split(".")[0]
    disclosures = dart.download_disclosures(stock_code=stock_code, kind="B")
    news_df = news.search_naver_news(COMPANY_NAME, display=20)
    news_title_col, news_desc_col = "title", "description"
else:
    disclosures = sec.download_filings(TICKER, forms=("8-K", "10-Q"))
    news_df = news.search_finnhub_news(TICKER, from_date="2026-06-01", to_date="2026-07-22")
    news_title_col, news_desc_col = "headline", "summary"

print(f"\n최근 중요 공시 {len(disclosures)}건, 뉴스 {len(news_df)}건 수집 완료")
disclosures.head()

현재가: 424.61 USD
시가총액: 2202228752384

최근 중요 공시 0건, 뉴스 241건 수집 완료


,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,isXBRLNumeric,primaryDocument,primaryDocDescription


In [3]:
from src.collector import transcript

transcript_df = None
if MARKET == "US":
    try:
        transcript_df = transcript.download_transcript(TICKER)
        print(f"실적발표 대본 {len(transcript_df)}개 문단 수집 완료 (분기: {transcript_df['fiscal_quarter'].iloc[0]})")
        display(transcript_df[["speaker", "speaker_role", "section", "text"]].head())
    except ValueError as e:
        print(f"대본을 못 찾았다({e}) — 이후 대본 관련 셀은 자동으로 건너뛴다. TICKER를 최근 실적발표한 다른 종목으로 바꿔도 된다.")
else:
    print("한국 종목은 실적발표 대본을 지원하지 않는다 (fool.com에 한국 기업 대본이 없음) — 이후 대본 관련 셀은 건너뛴다.")

실적발표 대본 154개 문단 수집 완료 (분기: Q2 2026)


,speaker,speaker_role,section,text
0,NaN,NaN,prepared_remarks,Image source: The Motley Fool.
1,NaN,NaN,prepared_remarks,"Thursday, July 16, 2026 at 2:00 a.m. ET"
2,NaN,NaN,prepared_remarks,Need a quote from a Motley Fool analyst? Email...
3,NaN,NaN,prepared_remarks,Management reported that second quarter result...
4,Jeff Su,IR,prepared_remarks,"Good afternoon, everyone, and welcome to TSMC'..."


## ② Document Processor + ③ Financial RAG

대본을 Q&A 경계를 인식하며 청킹하고(`chunking.py`), BGE-M3로 임베딩해(`embedding.py`) FAISS+BM25 하이브리드 인덱스에 넣는다(`vector_store.py`). 그다음 하이브리드 검색 → 리랭크 → 출처 부착까지(`retrieval.py`) 실제 질의로 확인한다.

(첫 실행 시 임베딩/리랭커 모델을 허깅페이스에서 내려받는다 — 수 분 소요될 수 있다.)

In [4]:
from src.rag.chunking import chunk_dataframe
from src.rag.embedding import embed_chunks
from src.rag.vector_store import VectorStore
from src.rag import retrieval

store = None
if transcript_df is not None:
    chunks = chunk_dataframe(
        transcript_df, text_col="text",
        metadata_cols=["speaker", "speaker_role", "section", "fiscal_quarter"],
        extra_metadata={"ticker": TICKER}, merge_adjacent=True, speaker_col="speaker",
        section_col="section", speaker_type_col="speaker_type",
    )
    embedded = embed_chunks(chunks)
    store = VectorStore(dim=len(embedded.iloc[0]["embedding"]))
    store.add(embedded)
    print(f"청크 {len(embedded)}개 인덱싱 완료")

    result = retrieval.search(store, f"What did {COMPANY_NAME} say about CapEx or guidance?", top_k=3, group_cols=("ticker",))
    for _, row in result.iterrows():
        print("\n===", row["citation"], "| rerank_score:", round(row["rerank_score"], 3))
        print(row["text"][:250])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

청크 56개 인덱싱 완료


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


=== TSM | Q2 2026 | Felix Pan, Jeff Su, C.C. Wei | Analyst | qa | chunk 145-147 | rerank_score: 0.973
Felix Pan: Hello, good afternoon. Thank you for taking my question. My first question is regarding to the CapEx revision. From year to date, TSMC raised the CapEx guidance by almost $10 billion. Can you give me some color where is the upside from how

=== TSM | Q2 2026 | Sunny Lin, Jeff Su, Wendell Huang | Analyst | qa | chunk 28-30 | rerank_score: 0.954
Sunny Lin: Thank you very much. Congrats on the very strong performance and outlook. Number one, I'll do a double click on the CapEx. Very encouraging CapEx outlook. I do think it's essential that TSMC showcase a stronger determination in capacity ex

=== TSM | Q2 2026 | Haas Liu, Jeff Su, C.C. Wei | Analyst | qa | chunk 110-115 | rerank_score: 0.93
Haas Liu: Yes. Thanks, C.C., Wendell, and Jeff for taking my questions, and congrats on the great results. My first question is regarding your CapEx and sales. You gave a pretty solid CapE

## ④ Earnings Call Analyzer

키워드 언급 횟수, 화자 역할별 발언, RAG 기반 주제 요약(Guidance/Risk)을 뽑는다.

In [5]:
from src.analytics.earnings_call_analyzer import count_keyword_mentions, extract_speaker_statements, summarize_topic

earnings_call_summaries = {}
if transcript_df is not None:
    print("키워드 언급 횟수:", count_keyword_mentions(transcript_df))

    ceo_statements = extract_speaker_statements(transcript_df, "CEO")
    print(f"\nCEO 발언 {len(ceo_statements)}건, 첫 발언: {ceo_statements[0][:150] if ceo_statements else 'N/A'}")

    # guidance/risk 2개뿐이면 Investment Thesis에 넘어가는 근거가 얕아진다 — CapEx/margin/demand까지
    # 늘려서 컨퍼런스콜에서 나온 실질적인 논의를 더 많이 반영한다.
    for topic in ["guidance", "risk", "capex", "margin", "demand"]:
        earnings_call_summaries[topic] = summarize_topic(store, topic, ticker=TICKER, top_k=4)
        print(f"\n[{topic}] {earnings_call_summaries[topic]['summary']}")

키워드 언급 횟수: {'AI': 45, 'CapEx': 47, 'Inventory': 3, 'Margin': 11, 'Demand': 53, 'Guidance': 13, 'Risk': 3}

CEO 발언 47건, 첫 발언: Thank you, Wendell. Good afternoon, everyone. First, let me start with our near-term demand outlook. We concluded our second quarter with revenue of $



[guidance] 제공된 발언 내용에 따른 가이던스 관련 요약은 다음과 같습니다.

TSMC의 2026년 2분기 매출은 가이던스 상단인 402억 달러를 달성했으며, 3분기 매출 가이던스는 446억 달러에서 458억 달러 사이로 제시되었습니다. 이는 3분기 매출 중간값 기준으로 전분기 대비 12%, 전년 동기 대비 37% 증가한 수치입니다. 아울러 회사는 강력한 AI 수요와 최첨단 공정에 힘입어 연간 가이던스를 상향 조정했으며, 2026년 미달러 기준 매출 성장률이 전년 대비 40%를 살짝 상회할 것으로 전망했습니다.



[risk] TSMC는 미래 예측 진술에 실제 결과와 차이를 유발할 수 있는 상당한 위험과 불확실성이 수반된다고 알렸습니다. 또한 부품 가격 상승과 거시경제적 불확실성으로 인해 소비자 및 가격 민감한 시장 부문이 어려움을 겪고 있어 신중한 경영 계획을 수립하고 있습니다. 아울러 2나노미터(nm) 생산 가속화 및 해외 제조 확장에 따른 마진 희석 요인이 존재하는 것으로 나타났습니다.



[capex] TSMC는 2분기에 4,960억 대만 달러(157억 달러)의 자본지출(CapEx)을 집행했으며, 강력한 AI 메가트렌드에 힘입어 향후 3년간의 CapEx가 지난 3년보다 훨씬 더 크게 증가할 것이라고 밝혔습니다. 2026년 CapEx 예산은 첨단 공정 기술에 70~80%, 특화 기술에 약 10%, 첨단 패키징·테스트·마스크 제작 등에 10~20%가 배정될 예정입니다. 또한 병목 현상 해결을 위해 전공정과 후공정 간 투자의 유연성을 유지하면서, 대규모 투자 속에서도 지속적인 수익성 성장과 주주 배당 확대를 이어나갈 계획입니다.



[margin] 2026년 2분기 매출총이익률은 비용 개선과 가동률 상승에 힘입어 67.7%를 기록했으나, 3분기에는 2nm 공정 양산에 따른 3~4%포인트의 마진 희석과 해외 공장 확장의 영향으로 매출총이익률 65%~67%(영업이익률 56%~58%) 수준으로 하락할 것으로 전망됩니다. 이러한 마진 희석 요인은 선단 공정에 대한 강력한 수요와 함께 생산성 향상, 공정 간 가동률 최적화 등 지속적인 비용 개선 노력을 통해 일부 상쇄될 예정입니다.



[demand] TSMC의 2026년 2분기 실적은 고성능 컴퓨팅(HPC), 생성형 AI 및 5G 메가트렌드에 기반한 첨단 공정 기술의 강력한 수요에 힘입어 호조를 보였습니다. 수요와 공급의 격차가 크게 벌어짐에 따라 회사는 대만, 미국, 일본의 신규 공장 가동을 가속화하고 연간 매출 전망과 설비투자(CapEx)를 상향 조정했습니다. 또한, 경영진은 AI 관련 수요가 지속적으로 강해지고 있다고 밝히면서도, 고객사 및 CSP들의 공격적인 수요 요청을 신중하게 평가·판단하여 대규모 투자 결정에 반영하고 있다고 설명했습니다.


## ⑤ News Analyzer

수집한 뉴스를 논조(Positive/Negative/Neutral)와 영향도(High/Medium/Low)로 분류한다 (Gemini 호출).

In [6]:
from src.analytics.news_analyzer import classify_news, summarize_classification

classified_news = classify_news(news_df, COMPANY_NAME, title_col=news_title_col, desc_col=news_desc_col)
news_summary = summarize_classification(classified_news, title_col=news_title_col)

print(news_summary["tone_counts"])
print(news_summary["impact_counts"])
print("\n고영향 부정 뉴스:")
for title in news_summary["high_impact_negative_titles"][:5]:
    print(" -", title)

{'Positive': 126, 'Neutral': 75, 'Negative': 40}
{'Low': 113, 'High': 70, 'Medium': 58}

고영향 부정 뉴스:
 - TSMC Beat, The Stock Fell Anyway: Here's What Got Priced In
 - Taiwan Semi’s AI Spending Spree Tests Investor Nerves as Capex Surges Past $60 Billion


## ⑥ Financial Analyzer

표준 재무비율(Margin, ROE, PER, PEG, EV/EBITDA, Debt/Equity, FCF, Revenue Growth)을 뽑는다. ROIC은 시도했다가 뺐다 — yfinance의 operatingMargins가 TTM이 아니라 최신 분기 단독치라(다른 필드는 TTM인 것과 달리) 시점이 안 맞아 값이 부풀려졌고(계산값 70.3% vs 실제 재무제표 기준 54.2%), 바로잡으려면 분기별 재무제표를 별도로 받아야 해서 지표 하나의 정밀도 대비 비용이 크다고 판단했다 (`financial_analyzer.py` 독스트링 참고).

In [7]:
from src.analytics.financial_analyzer import summarize

financial_ratios = summarize(info)
for k, v in financial_ratios.items():
    print(f"{k}: {v}")

gross_margin: 0.6423
operating_margin: 0.60344005
net_margin: 0.49923
per: 34.97611
peg: 1.01
ev_to_ebitda: 4.763
debt_to_equity: 15.174
roe: 0.39969003
roa: 0.19014
revenue_growth: 0.36
fcf: 22791440399.9292


## ⑦ Valuation Engine

DCF, Comparable Company Analysis, Sensitivity, Football Field을 계산한다. `growth_rate`/`peer_tickers`는 실제 애널리스트가 조정하는 값이니 자유롭게 바꿔서 실험해봐도 된다.

In [8]:
from src.valuation.dcf import compute_wacc, get_equity_risk_premium, get_growth_rate_estimate, get_risk_free_rate, run_dcf

RISK_FREE_RATE = get_risk_free_rate(MARKET)  # 미국은 yfinance ^TNX로 실시간 조회, 한국은 yfinance에 국채수익률 티커가 없어 확인 시점 값 사용(dcf.py 참고)
EQUITY_RISK_PREMIUM = get_equity_risk_premium(MARKET)  # Damodaran(NYU Stern) 데이터 기준 — 한국은 국가리스크프리미엄이 더해져 미국보다 높다
TAX_RATE = 0.21 if MARKET == "US" else 0.275  # 한국은 2026년 기준 대기업(과세표준 3천억원 초과) 실효세율: 국세 25% + 지방소득세 2.5%
GROWTH_RATE = get_growth_rate_estimate(info)  # yfinance의 revenueGrowth(최근 TTM 매출 성장률) — 하드코딩된 임의값 대신 실제 데이터 기반(dcf.py 참고)

wacc = compute_wacc(info, risk_free_rate=RISK_FREE_RATE, equity_risk_premium=EQUITY_RISK_PREMIUM, tax_rate=TAX_RATE)
net_debt = (info.get("totalDebt") or 0) - (info.get("totalCash") or 0)

dcf_result = run_dcf(
    base_fcf=info["freeCashflow"], wacc=wacc, growth_rate=GROWTH_RATE, terminal_growth_rate=0.03,
    years=5, net_debt=net_debt, shares_outstanding=info["sharesOutstanding"],
)
print("risk-free rate:", f"{RISK_FREE_RATE:.2%}", "| ERP:", f"{EQUITY_RISK_PREMIUM:.2%}", "| growth rate:", f"{GROWTH_RATE:.2%}", "| WACC:", f"{wacc:.1%}")
for k, v in dcf_result.items():
    print(f"{k}: {v:,.2f}")

risk-free rate: 4.65% | ERP: 4.45% | growth rate: 36.00% | WACC: 10.1%
enterprise_value: 1,170,514,446,662.57
equity_value: 1,248,707,503,079.85
pv_fcf: 224,306,847,550.17
pv_terminal_value: 946,207,599,112.39
implied_share_price: 240.76


In [9]:
from src.valuation.comps import PeerSelectionError, comps_valuation, find_peer_tickers

# 예전엔 피어 티커를 사람이 직접 골라 하드코딩했다 — 이제 Gemini 검색으로 사업모델이 유사한 경쟁사를
# 찾고, yfinance로 실존 여부를 교차검증해 자동 선정한다(comps.py 참고). 검증된 피어가 2개 미만이면
# 중앙값 배수 산출 자체가 불가능하므로 PeerSelectionError가 난다 — 이 경우 Comps는 건너뛴다.
comps_result = None
try:
    PEER_TICKERS = find_peer_tickers(COMPANY_NAME, TICKER)
    print(f"자동 선정된 피어그룹: {PEER_TICKERS}")

    peer_infos = {peer: market.get_company_info(peer) for peer in PEER_TICKERS}
    comps_result = comps_valuation(info, peer_infos)

    display(comps_result["peer_table"])
    print("median EV/EBITDA:", comps_result["median_ev_ebitda"], "| median PER:", comps_result["median_pe"])
    print("내재주가 (EV/EBITDA):", comps_result["implied_price_from_ev_ebitda"])
    print("내재주가 (PER):", comps_result["implied_price_from_pe"])
except PeerSelectionError as e:
    print(f"피어그룹 자동 선정 실패({e}) — Comps 밸류에이션을 건너뛴다.")

자동 선정된 피어그룹: ['GFS', 'UMC', 'TSEM', '000990.KS', '005930.KS']


,enterpriseToEbitda,trailingPE
ticker,,
GFS,15.419,42.791367
UMC,-0.047,34.282257
TSEM,50.034,115.104164
000990.KS,8.411,NaN
005930.KS,11.389,NaN


median EV/EBITDA: 11.389 | median PER: 42.791367
내재주가 (EV/EBITDA): 230.09717191925154
내재주가 (PER): 519.48719538


In [10]:
from src.valuation.sensitivity import dcf_sensitivity_table
from src.valuation.football_field import build_football_field

sensitivity = dcf_sensitivity_table(
    base_fcf=info["freeCashflow"], growth_rate=GROWTH_RATE, years=5, net_debt=net_debt,
    shares_outstanding=info["sharesOutstanding"],
    wacc_range=[wacc - 0.03, wacc - 0.015, wacc, wacc + 0.015, wacc + 0.03],
    terminal_growth_range=[0.02, 0.03, 0.04, 0.05],
)
display(sensitivity.round(1))

football_field_inputs = {
    "DCF (Bear-Bull)": (sensitivity.min().min(), sensitivity.max().max()),
    "52주 가격범위": (info["fiftyTwoWeekLow"], info["fiftyTwoWeekHigh"]),
}
if comps_result is not None:
    football_field_inputs["Comps (EV/EBITDA, PE)"] = tuple(sorted(
        [comps_result["implied_price_from_ev_ebitda"], comps_result["implied_price_from_pe"]]
    ))

football_field = build_football_field(football_field_inputs)
display(football_field)
print("현재가:", info["currentPrice"])

terminal_growth_rate,0.02,0.03,0.04,0.05
wacc,,,,
0.071263,351.0,424.4,544.7,778.3
0.086263,268.5,307.9,364.3,451.8
0.101263,216.8,240.8,272.6,316.9
0.116263,181.4,197.2,217.3,243.3
0.131263,155.7,166.8,180.3,197.1


,method,low,high,range,midpoint
0,52주 가격범위,223.700000,479.000000,255.300000,351.350000
1,"Comps (EV/EBITDA, PE)",230.097172,519.487195,289.390023,374.792184
2,DCF (Bear-Bull),155.724212,778.258914,622.534702,466.991563


현재가: 424.61


## ⑧ Investment Thesis Generator

지금까지 계산한 모든 근거(Valuation, Financials, Earnings Call, News)를 LLM에 넘겨 Investment Thesis/Bear Case/Catalyst/Risk/Target Price를 생성한다. Target Price는 위 Football Field 범위 안에서 산출하도록 프롬프트에서 강제한다.

In [11]:
from src.agents.thesis_agent import generate_thesis

context = {
    "valuation": {
        "dcf_range": [float(sensitivity.min().min()), float(sensitivity.max().max())],
        "comps_range": (
            sorted([comps_result["implied_price_from_ev_ebitda"], comps_result["implied_price_from_pe"]])
            if comps_result is not None else None
        ),
        "52w_range": [info["fiftyTwoWeekLow"], info["fiftyTwoWeekHigh"]],
        "current_price": info["currentPrice"],
    },
    "financials": financial_ratios,
    "earnings_call": {k: v["summary"] for k, v in earnings_call_summaries.items()},
    "news": news_summary,
}

thesis = generate_thesis(COMPANY_NAME, TICKER, context)
print("Investment Thesis:")
for point in thesis["investment_thesis"]:
    print(" -", point)
print("\nBear Case:")
for point in thesis["bear_case"]:
    print(" -", point)
print("\nCatalyst:")
for point in thesis["catalysts"]:
    print(" -", point)
print("\nRisk:")
for point in thesis["risks"]:
    print(" -", point)
print("\nTarget Price:", thesis["target_price"], "|", thesis["target_price_rationale"])

Investment Thesis:
 - HPC, 생성형 AI 및 5G 메가트렌드에 기반한 강력한 첨단 공정 수요로 2026년 미달러 기준 연간 매출 성장률이 40%를 상회할 것으로 전망됨
 - 매출총이익률 64.23%, ROE 39.97% 등 뛰어난 수익성 및 잉여현금흐름(FCF) 창출력을 바탕으로 대규모 투자 속에서도 지속적인 주주 배당 확대 추진
 - 2나노미터(nm) 생산 가속화와 대만, 미국, 일본의 신규 공장 가동을 통한 글로벌 첨단 파운드리 공급 능력 및 시장 지배력 강화

Bear Case:
 - 2nm 공정 양산에 따른 3~4%p 수준의 마진 희석 및 해외 제조 시설 확장에 따른 수익성 압박
 - 600억 달러를 초과하는 대규모 CapEx 지출 급증이 투자자 불안을 자극하여 호실적 발표에도 주가 상방을 제한할 가능성
 - 부품 가격 상승 및 거시경제 불확실성에 따른 소비자 및 가격 민감 시장 부문의 경기 둔화 위험

Catalyst:
 - 2026년 3분기 매출 가이던스(전년 동기 대비 37% 증가한 446억~458억 달러) 달성 및 AI 관련 가이던스 추가 상향 가능성
 - 미국, 일본 등 해외 신규 공장 및 첨단 패키징·테스트 라인의 본격적인 가동 가속화

Risk:
 - 2nm 공정 초기 양산 및 해외 확장 과정에서 발생하는 비용 부담과 마진 하락 위험
 - 거시경제적 불확실성 및 대규모 설비투자로 인한 자본 효율성 저하 우려

Target Price: 430.11 | DCF 밸류에이션(60%)과 Comps 상대가치 평가(40%)의 가중평균으로 산출된 기준점(Weighted Anchor)인 430.11달러를 목표주가로 적용하였습니다.


## ⑨ Portfolio Monitor

공시/뉴스/실적발표/목표주가 변동 네 가지를 체크한다. 처음 실행하는 종목이면 저장된 이전 상태가 없어 전부 "이상"으로 잡힌다 — 바로 다시 실행해보면 두 번째부터는 트리거가 꺼지는 걸 볼 수 있다.

In [12]:
from src.agents.portfolio_monitor import run_daily_check

monitor_result = run_daily_check(TICKER, cheap_target_price=thesis["target_price"], classified_news_df=classified_news)
print("트리거:", monitor_result["triggers"])
print("전체 재생성 필요:", monitor_result["need_full_regeneration"])
print("신규 중요공시:", len(monitor_result["new_disclosures"]))

트리거: {'material_disclosure': False, 'high_impact_news': False, 'new_earnings_call': False, 'target_price_drift': True}
전체 재생성 필요: True
신규 중요공시: 0


## ⑩ PDF Report

지금까지의 모든 결과를 조합해 최종 PDF 투자보고서를 생성한다.

In [13]:
from src.report.report_generator import build_context, generate_business_analysis, generate_company_overview, generate_pdf

# ①~⑨ 어디서도 회사 개요/사업분석 텍스트를 만들지 않으므로(사업보고서/10-K 본문 요약은 별도 컴포넌트가
# 필요한 영역), yfinance info에 이미 있는 데이터만 근거로 Gemini가 한글로 생성한다 — 원문/데이터에
# 없는 내용은 추측하지 않도록 프롬프트에서 강제한다(report_generator.py 참고).
company_overview = generate_company_overview(info)
business_analysis = generate_business_analysis(info, financial_ratios, comps_result=comps_result)

report_context = build_context(
    company_name=COMPANY_NAME, ticker=TICKER, currency=info.get("currency", "USD"),
    current_price=info["currentPrice"], financial_ratios=financial_ratios,
    dcf_result=dcf_result, comps_result=comps_result, football_field_table=football_field,
    thesis=thesis, news_summary=news_summary, earnings_call_summaries=earnings_call_summaries,
    fiscal_quarter=transcript_df["fiscal_quarter"].iloc[0] if transcript_df is not None else None,
    company_overview=company_overview, business_analysis=business_analysis,
)
output_path = generate_pdf(report_context)
print("PDF 생성 완료:", output_path)

PDF 생성 완료: C:\Users\vkdlf\Desktop\projects\ai-equity-research-agent\output\reports\TSM_20260723.pdf


## Bonus: Tool-calling Agent

지금까지 위에서 손으로 순서대로 실행한 파이프라인을, `research_agent.py`의 에이전트에게 자연어로 질문만 던지면 LLM이 알아서 필요한 도구(`tools.py`)를 골라 호출한다. `planner.py`에 tool-calling 루프(판단→실행→결과반영→반복)를 프레임워크 없이 직접 구현했다 — 자세한 이유는 해당 모듈 독스트링 참고.

질문을 자유롭게 바꿔서 실험해봐도 된다. 간단한 질문은 도구 1개, 복잡한 질문은 여러 도구를 체이닝하는 걸 호출 로그로 확인할 수 있다.

In [14]:
from src.agents.research_agent import run_equity_research_agent

answer, call_log = run_equity_research_agent(f"{TICKER} 최근 재무비율이랑 뉴스 논조 같이 알려줘")
print("=== 답변 ===")
print(answer)
print("\n=== 호출된 도구 ===")
for entry in call_log:
    print(" -", entry["tool"], entry["args"])

=== 답변 ===
TSMC(티커: TSM)의 최근 주요 재무비율과 최근 뉴스 논조 요약 결과입니다.

---

### 1. 주요 재무비율 (`get_financial_ratios` 조회 결과)

* **성장성 및 수익성**
  * **매출성장률 (Revenue Growth):** 36.00%
  * **총이익률 (Gross Margin):** 64.23%
  * **영업이익률 (Operating Margin):** 60.34%
  * **순이익률 (Net Margin):** 49.92%
  * **ROE (자기자본이익률):** 39.97%
  * **ROA (총자산이익률):** 19.01%

* **밸류에이션 및 재무 건전성**
  * **PER (주가수익비율):** 34.98배
  * **PEG Ratio:** 1.01
  * **EV/EBITDA:** 4.76배
  * **부채비율 (Debt to Equity):** 15.17%
  * **자유현금흐름 (FCF):** 약 227억 8,863만 달러 ($22,788,628,000.44)

---

### 2. 최근 뉴스 논조 요약 (`get_news_tone_summary` 조회 결과)

* **논조 분포 (Tone Counts)**
  * **Positive (긍정적):** 70건
  * **Neutral (중립적):** 82건
  * **Negative (부정적):** 12건
  * *전반적으로 긍정적 및 중립적인 뉴스가 압도적인 비중을 차지하고 있습니다.*

* **영향도 분포 (Impact Counts)**
  * **High (높음):** 46건
  * **Medium (중간):** 36건
  * **Low (낮음):** 82건

* **주요 부정적 뉴스 이슈 (High-Impact Negative)**
  * 자본지출(Capex) 증가 우려: 설비투자가 600억 달러를 넘어서는 등 AI 관련 지출 급증이 투자자 심리에 부담을 줄 수 있다는 뉴스가 보도되었습니다.
    * 예시 헤드라인: *"Ta